# Social Media Posting Strategy — Explanatory Model

---

## 1. Problem Framing

### Business Problem

Hearth Haven's social media team posts without a systematic understanding of
which combinations of platform, content topic, timing, and format are most
strongly associated with high engagement. They make decisions based on intuition
and trial and error, with no structured feedback loop.

This pipeline answers the question: **Which controllable post characteristics
are most strongly associated with higher engagement rates, and by how much?**

The deployed output is a **Recommendation Panel** on the social media management
dashboard, showing the highest-performing combinations of platform, content topic,
post type, and posting time — giving managers a data-grounded content strategy
guide rather than a blank canvas.

### Who Cares About This

- **Social media managers** — need concrete, actionable posting guidance they can
  apply immediately without a marketing background.
- **Organization leadership** — wants to maximize reach and donor pipeline from
  social media without hiring a dedicated marketing team.

### Predictive vs. Explanatory

This pipeline uses an **explanatory approach**. Rather than predicting a score
for a specific planned post (that is `socials_pred_engagement_amount`), here we
want to understand *which post characteristics matter and by how much* — producing
a general strategy guide rather than per-post predictions.

OLS coefficients are the primary output. Each coefficient answers: "Holding all
other post characteristics constant, how much does this attribute change the
expected engagement rate?"

### Success Metrics

- **Primary:** Coefficient significance (p-values) and magnitude
- **Secondary:** Adjusted R² — how much of engagement variance the model explains
- **Not primary:** Out-of-sample predictive accuracy — use `socials_pred_engagement_amount`
  for that

### A Note on Causation

Even significant coefficients here are associations, not causal effects. A post
type that is associated with higher engagement may simply reflect that the
organization already knows which content works and produces more of it —
reverse causation. The coefficients tell us what has historically co-occurred
with high engagement, not what will definitely cause it if we change our strategy.

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

sys.path.append(os.path.dirname(os.path.abspath('.')))
os.chdir('..')

from functions.fn_domain_prep import prepare_social_media
from functions.fn_prepare import define_features, split_data
from functions.fn_model_causal import (
    fit_causal_regression,
    get_coefficients,
    check_assumptions,
    check_vif,
    refit_with_robust_se,
    run_stepwise_pvalue,
)

print("All imports successful.")

### 2.1 Load and Prepare Data

`prepare_social_media()` encodes every cleaning and feature engineering decision
from `eda_social_media.ipynb`. It tries Azure SQL first and falls back to CSV.

**Tables joined:** `social_media_posts`, `donations`

**Key preparation decisions encoded:**
- Structural columns dropped: `platform_post_id`, `post_url`, `caption`, `hashtags`
- Temporal features engineered: `post_month`, `post_is_weekend` from `created_at`
- Intentional nulls filled with domain-appropriate sentinels
- Outliers capped and skew transformed: `impressions`, `reach`
- `engagement_rate` is the target: total interactions / reach

In [ ]:
df, NUMERIC, CATEGORICAL, DROP = prepare_social_media()

TARGET = 'engagement_rate'

print(f"Dataset shape: {df.shape}")
print(f"Target mean: {df[TARGET].mean():.4f}  |  Std: {df[TARGET].std():.4f}")

### 2.2 Feature Definition

`define_features()` is called with `DROP['engagement_rate']`. All post-publication
metrics are excluded — we can only use features available *before* a post goes live
to produce actionable posting guidance.

In [ ]:
X, y = define_features(
    df,
    target=TARGET,
    numeric=NUMERIC,
    categorical=CATEGORICAL,
    drop_cols=DROP[TARGET],
)

categorical_in_X = [c for c in CATEGORICAL if c in X.columns]
numeric_in_X     = [c for c in NUMERIC     if c in X.columns]
X[categorical_in_X] = X[categorical_in_X].astype(str).replace({'nan': np.nan, '<NA>': np.nan})

print(f"Feature matrix: {X.shape[0]} rows × {X.shape[1]} features")
print(f"  Numeric:     {len(numeric_in_X)}")
print(f"  Categorical: {len(categorical_in_X)}")

### 2.3 Exploratory Confirmation

These cells document the raw associations before modeling — giving context for
interpreting the OLS coefficients in Section 3.

In [ ]:
# Mean engagement by each categorical feature — the raw signal
for col in ['platform', 'post_type', 'content_topic', 'sentiment_tone',
            'media_type', 'has_call_to_action', 'features_resident_story']:
    if col in X.columns:
        rate = (pd.concat([X[col], y], axis=1)
                  .groupby(col)[TARGET]
                  .agg(['mean', 'std', 'count'])
                  .rename(columns={'mean': 'mean_eng', 'std': 'std_eng', 'count': 'n'})
                  .sort_values('mean_eng', ascending=False))
        print(f"Engagement by {col}:")
        print(rate.round(4).to_string(), "\n")

In [ ]:
# Top numeric correlations
corr = X[numeric_in_X].corrwith(y).sort_values(key=abs, ascending=False)
print("Numeric features by |correlation| with engagement_rate:")
print(corr.round(4).to_string())

In [ ]:
# Heatmap: mean engagement by platform × content_topic
if 'platform' in X.columns and 'content_topic' in X.columns:
    pivot = (pd.concat([X[['platform', 'content_topic']], y], axis=1)
               .groupby(['platform', 'content_topic'])[TARGET]
               .mean()
               .unstack(fill_value=0))

    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_yticks(range(len(pivot.index)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(pivot.index, fontsize=8)
    plt.colorbar(im, ax=ax, label='Mean Engagement Rate')
    plt.title('Mean Engagement Rate by Platform × Content Topic')
    plt.tight_layout()
    plt.show()

---
## 3. Causal Model Specification

### 3.1 Train/Test Split

We hold out a test set for an honest generalization check in Section 4, but feature
selection and coefficient interpretation use the training set only.

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, stratify=False)

### 3.2 Multicollinearity Check (VIF)

High VIF (> 10) indicates collinear features whose coefficients will be unstable
and hard to interpret. We drop iteratively before fitting OLS.

In [ ]:
X_train_enc = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_train_enc = X_train_enc.apply(pd.to_numeric, errors='coerce').fillna(0)

print(f"Encoded matrix: {X_train_enc.shape[0]} rows × {X_train_enc.shape[1]} columns")

vif_df   = check_vif(X_train_enc, threshold=10.0)
high_vif = vif_df[vif_df['VIF'] > 10]['feature'].tolist()            if 'VIF' in vif_df.columns else []

print(f"\nFeatures with VIF > 10: {high_vif if high_vif else 'None'}")
X_clean = X_train_enc.drop(columns=high_vif, errors='ignore')
print(f"Matrix after VIF cleanup: {X_clean.shape[1]} features remaining")

### 3.3 Initial OLS Fit

In [ ]:
results = fit_causal_regression(X_clean, y_train)
print(results.summary())

### 3.4 OLS Assumption Checks

We check heteroscedasticity specifically — engagement rate data often shows
non-constant variance (higher-engagement posts have more variable outcomes).
If homoscedasticity fails, we apply HC3 robust standard errors.

In [ ]:
verdicts = check_assumptions(results)

if verdicts.get('homoscedasticity', {}).get('verdict') != 'PASS':
    print("\n[ACTION] Homoscedasticity failed — applying HC3 robust standard errors")
    results = refit_with_robust_se(results)
    print("HC3 applied. Coefficients unchanged; standard errors and p-values updated.")

### 3.5 Stepwise p-value Feature Selection

We use stepwise backward elimination to remove non-significant features, producing
a parsimonious model where every remaining feature is significant at p < 0.05.
For OLS regression this gives us a clean, defensible posting strategy guide.

In [ ]:
results_final, kept_features = run_stepwise_pvalue(
    X_clean, y_train,
    p_threshold=0.05,
    model_type='linear',
)

print(f"\nFinal model: {len(kept_features)} significant features")
print(results_final.summary())

---
## 4. Evaluation & Interpretation

### 4.1 Coefficient Table

In [ ]:
coef_df = get_coefficients(results_final, model_type='linear')

print("All significant coefficients (sorted by magnitude):")
print(coef_df[['feature', 'coefficient', 'std_err', 'p_value', 'significant']]
      .to_string(index=False))

# Save for the dashboard
os.makedirs('models', exist_ok=True)
coef_df.to_csv('models/posting_strategy_coefficients.csv', index=False)
print("\nCoefficient table saved: models/posting_strategy_coefficients.csv")

In [ ]:
# Visualize
if len(coef_df) > 0:
    plot_df = coef_df.set_index('feature')['coefficient'].sort_values()
    colors  = ['coral' if v < 0 else 'steelblue' for v in plot_df]
    plot_df.plot(kind='barh',
                 figsize=(11, max(5, len(plot_df) * 0.4)),
                 color=colors)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title('Posting Strategy Coefficients\n'
              '(Positive = associated with higher engagement)', fontsize=12)
    plt.xlabel('Coefficient (engagement rate units)')
    plt.tight_layout()
    plt.show()

### 4.2 Model Fit

In [ ]:
print("Model fit statistics:")
print(f"  R²:           {results_final.rsquared:.4f}")
print(f"  Adjusted R²:  {results_final.rsquared_adj:.4f}")
print(f"  F-statistic:  {results_final.fvalue:.4f}  (p = {results_final.f_pvalue:.6f})")
print(f"  Observations: {int(results_final.nobs)}")

# Test set check
import statsmodels.api as sm
X_test_enc = pd.get_dummies(X_test, drop_first=True, dtype=int)
X_test_enc = X_test_enc.apply(pd.to_numeric, errors='coerce').fillna(0)
X_test_final = X_test_enc.reindex(columns=kept_features, fill_value=0)
X_test_const = sm.add_constant(X_test_final, has_constant='add')
y_pred_test  = results_final.predict(X_test_const)

from sklearn.metrics import r2_score
print(f"\nTest set R²: {r2_score(y_test, y_pred_test):.4f}")

### 4.3 Posting Strategy Interpretation

**Reading the coefficients:**

Each coefficient represents the estimated change in engagement rate associated
with that post characteristic, holding all others constant. Multiply by 100 to
read as percentage points.

**Recommended posting strategy (based on significant positive coefficients):**

The coefficients directly translate into a posting guide the social media team
can follow. Group the significant positive features into an actionable checklist:

- **Content:** Features associated with highest engagement (likely `features_resident_story`,
  `content_topic` values like Reintegration or DonorImpact, `sentiment_tone`
  Emotional or Hopeful)
- **Format:** Media types and post types most associated with engagement
  (likely Reels, Video, ImpactStory)
- **Timing:** Day of week and posting hour coefficients
- **Platform:** Which platforms show the strongest positive associations

**What to avoid** (negative coefficients):

Features with significant negative coefficients represent post characteristics
associated with below-average engagement — worth specifically avoiding or
minimizing.

### 4.4 Limitations

- **Correlation, not causation.** The org may already intuitively know which
  content works and produce more of it — reverse causation cannot be ruled out.
- **OLS linearity assumption.** The predictive model (GradientBoosting, R²=0.75)
  outperforms OLS (R²≈0.35) because the true relationships are nonlinear. The
  OLS coefficients are approximations of the dominant linear trends.
- **Platform × content interactions.** A Reel on Instagram may vastly outperform
  the sum of individual coefficients. The recommendation panel should be understood
  as a guide for each dimension independently, not a guaranteed formula.

---
## 5. Deployment

This is an explanatory pipeline. The coefficient table is served as a static GET endpoint — no inference server needed.

In [ ]:
# Coefficient table already saved in Section 4.1
# Generate a structured recommendation summary for the dashboard

sig = coef_df[coef_df['p_value'] < 0.05].copy()
positive = sig[sig['coefficient'] > 0].sort_values('coefficient', ascending=False)
negative = sig[sig['coefficient'] < 0].sort_values('coefficient')

summary = {
    'top_positive_factors': positive[['feature', 'coefficient']].head(5).to_dict('records'),
    'top_negative_factors': negative[['feature', 'coefficient']].head(3).to_dict('records'),
    'model_r2':             round(results_final.rsquared, 4),
    'model_adj_r2':         round(results_final.rsquared_adj, 4),
    'n_observations':       int(results_final.nobs),
    'model_version':        'posting_strategy_v1',
}

import json
with open('models/posting_strategy_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Strategy summary saved: models/posting_strategy_summary.json")
print(json.dumps(summary, indent=2))

---
## 6. API Response Reference

```json
GET /api/analysis/posting-strategy

{
  "top_positive_factors": [
    { "feature": "string", "coefficient": "float", "label": "string" }
  ],
  "top_negative_factors": [
    { "feature": "string", "coefficient": "float", "label": "string" }
  ],
  "best_combinations": [
    {
      "platform": "string",
      "content_topic": "string",
      "sentiment_tone": "string",
      "estimated_lift": "float"
    }
  ],
  "model_version": "posting_strategy_v1",
  "generated_at": "ISO datetime"
}
```

**top_positive_factors** — The post characteristics most strongly associated
with above-average engagement, sorted by coefficient magnitude. Used to populate
the "Do More Of This" section of the recommendation panel.

**top_negative_factors** — Characteristics associated with below-average
engagement. Used to populate the "Avoid" section.

**best_combinations** — Pre-computed top 5 platform × content_topic × sentiment_tone
combinations by estimated engagement lift (sum of their coefficients). Gives
managers a concrete shortlist of high-performing post templates.

**estimated_lift** — Sum of the individual coefficients for each feature in the
combination, relative to the baseline engagement rate. A lift of +0.03 means
approximately 3 percentage points above the average engagement rate.

---
### .NET Endpoint to add to an appropriate controller

```csharp
[HttpGet("posting-strategy")]
[Authorize]
public IActionResult GetPostingStrategy()
{
    var summaryPath = Path.Combine(_env.ContentRootPath,
                                   "models", "posting_strategy_summary.json");
    if (!System.IO.File.Exists(summaryPath))
        return NotFound(new { error = "Posting strategy model not found" });

    var json = System.IO.File.ReadAllText(summaryPath);
    return Content(json, "application/json");
}
```

*No `endpoints.py` or `server.py` changes needed — the .NET backend serves the
precomputed JSON summary directly.*

---
*Hearth Haven — IS 455 INTEX Pipeline*